In [ ]:
import json
import re
import copy
from tqdm import tqdm

# download train.jsonl and test.jsonl from https://huggingface.co/datasets/stanfordnlp/nnetnav-live/viewer/default/test?views%5B%5D=test&row=12

In [ ]:

def group_data(file_path):
    """
    Processes a JSONL-formatted string to extract and display specific
    information from consecutive lines based on their 'id'.

    Args:
        jsonl_string (str): A string containing JSONL data.
    """
    
    # Dictionary to store data, grouped by 'id'
    grouped_data = {}
    with open(file_path, 'r') as f:
        for line in tqdm(f):
            if line.strip():  # Skip empty lines
                data = json.loads(line)
                task_id = data['id']
                # if 'openweb_1655' == task_id:
                #     # Group data by 'id'
                if task_id not in grouped_data:
                    grouped_data[task_id] = []
                grouped_data[task_id].append(data)

    # Iterate through each group of tasks
    grouped_data2 = {}
    for task_id, task_lines in tqdm(grouped_data.items()):
        # print(f"--- Processing Task ID: {task_id} ---")
        grouped_data2[task_id] = {'objective': '', 'steps': []}
        # We need to loop up to the second to last element to access the next line
        for i in range(len(task_lines) - 1):
            current_line = task_lines[i]
            next_line = task_lines[i+1]
            
            # Extract user and assistant content from the current line
            user_content = next((item['content'] for item in current_line['messages'][1:] if item['role'] == 'user'), None)
            assistant_content = next((item['content'] for item in current_line['messages'][1:] if item['role'] == 'assistant'), None)
            
            # Extract user content from the next line
            next_user_content = next((item['content'] for item in next_line['messages'][1:] if item['role'] == 'user'), None)
            
            # Use a regular expression to extract key information from the user's content
            if user_content:
                objective_match = re.search(r"OBJECTIVE: (.*)", user_content)
                observation_match = re.search(r"OBSERVATION:\n(.*?)(?=\nOBJECTIVE:)", user_content, re.DOTALL)
                previous_actions_match = re.search(r"PREVIOUS ACTIONS:\n(.*)", user_content, re.DOTALL)

                objective = objective_match.group(1).strip() if objective_match else "Not Found"
                current_observation = observation_match.group(1).strip() if observation_match else "Not Found"
                previous_actions = previous_actions_match.group(1).strip() if previous_actions_match else "Not Found"
            
            # Extract the current action from the assistant's content
            current_action = "Not Found"
            if assistant_content:
                action_match = re.search(r"the next action I will perform is\s+(.*)", assistant_content, re.DOTALL)
                if action_match:
                    # Clean up the action string
                    current_action = action_match.group(1).strip()
                    current_action = re.sub('`', '', current_action)
                    current_action = current_action.strip()
            
            # Extract the observation from the next line's user content
            next_observation = "Not Found"
            if next_user_content:
                next_observation_match = re.search(r"OBSERVATION:\n(.*?)(?=\nOBJECTIVE:)", next_user_content, re.DOTALL)
                next_observation = next_observation_match.group(1).strip() if next_observation_match else "Not Found"
                
            # Print the results in the requested format
            # print(f"  OBJECTIVE: {objective}")
            # print(f"  current OBSERVATION: {current_observation[:200]}")
            # print(f"  previous ACTION: {previous_actions}")
            # print(f"  current ACTION: {current_action}")
            # print(f'assistant_content:{assistant_content}')
            # print(f"  next OBSERVATION: {next_observation[:200]}")
            # print("-" * 50)
            if not grouped_data2[task_id]['objective']:
                grouped_data2[task_id]['objective'] = objective
            grouped_data2[task_id]['steps'].append({'current_observation':current_observation,
                                                    'previous_actions': previous_actions,
                                                    'current_action': current_action,
                                                    'current_assistant_content': assistant_content,
                                                    'next_observation': next_observation,})
        # break
    return grouped_data2


def normalize_action_string(action_str, keep_press_enter_after=False, lower=True):
    """
    Cleans and normalizes an action string for comparison.
    This handles extra text, comments, and varying whitespace.
    
    Examples:
    "2: click [51] where [51] is " -> "click [51]"
    "click [50]" -> "click [50]"
    "type [123] ["my_content"      ] [0]" -> "type [123] ["my_content"]"
    """
    # Define a list of all valid action keywords
    action_keywords = [
        'click', 'type', 'hover', 'press', 'scroll',
        'new_tab', 'tab_focus', 'close_tab', 'go_back'
    ]

    # Use a regex to find the start of a valid action
    pattern = r'(' + '|'.join(action_keywords) + r')'
    if lower:
        action_str = action_str.lower()
    
    match = re.search(pattern, action_str)
    
    if not match:
        return ""

    # Extract the part of the string from the matched keyword onwards
    cleaned_str = action_str[match.start():].strip()
    
    # Remove any trailing comments or irrelevant text. This is a common pattern
    # where an action is followed by a description. We stop at the first non-valid
    # character sequence, which is typically a space followed by a word.
    # The regex below captures the action and its parameters (e.g., [id], content).
    
    # Updated regex pattern to capture the full action string and its arguments
    press_enter_pattern = r'\s*\[\d+\]' if keep_press_enter_after else ''
    action_pattern = (
        r'^(click\s+\[\d+\]|'
        r'type\s+\[\d+\]\s+\[.*?\]' + press_enter_pattern + '|'
        r'hover\s+\[\d+\]|'
        r'press\s+\[.*?\]|'
        r'scroll\s+(?:down|up)|'
        r'new_tab|'
        r'tab_focus\s+\[\d+\]|'
        r'close_tab|'
        r'go_back(\s+\[\d+\])?'
        r').*$'
    )             
    action_match = re.match(action_pattern, cleaned_str)
    
    if action_match:
        # If a valid action is matched, return only that part
        final_str = action_match.group(1)
        # Normalize whitespace (replace multiple spaces with a single space)
        final_str = re.sub(r'\s+', ' ', final_str)
        final_str = re.sub(r' \]', ']', final_str)
        # scroll down to scroll [down]
        final_str = re.sub(r'scroll (down|up)', r'scroll [\1]', final_str)
        return final_str.strip()
    
    # Fallback if the full action pattern doesn't match perfectly
    # This might happen for simple actions like 'new_tab'
    # In this case, we just normalize whitespace and trim.
    
    cleaned_str = re.sub(r'\s+', ' ', cleaned_str)
    if not cleaned_str:
        print('empty cleaned_str')
        assert True
    cleaned_str = re.sub(r' \]', ']', cleaned_str)
    return cleaned_str.strip()

def de_noise(grouped_data):
    grouped_data2 = copy.deepcopy(grouped_data)
    for task_id, task_item in tqdm(grouped_data2.items()):
        steps = task_item['steps']
        for i in range(0, len(steps)-1):
            step = steps[i]
            current_action = step['current_action']
            normalize_current_action = normalize_action_string(current_action)

            previous_actions = steps[i+1]['previous_actions'].strip().split('\n') # next step
            previous_action = re.sub(r'\d+:', '' , previous_actions[-1]).strip()
            normalize_previous_action = normalize_action_string(previous_action)

            if normalize_previous_action != normalize_current_action:
                # print(f'not equal {current_action} to {previous_action}')
                if valid_action(previous_action, step['current_observation']):
                    grouped_data2[task_id]['steps'][i]['current_action'] = normalize_action_string(previous_action, keep_press_enter_after=True, lower=False)
                    # print(f'change')
                # else:
                #     print(f"{task_id}-{i}, {previous_action} is not valid")

    return grouped_data2



def valid_action(action, observation):
    def extract_content(text):
        """
        Finds the first instance of text enclosed in square brackets that can be
        converted to a number and returns it.
        """
        # This regular expression finds all instances of text enclosed in square brackets
        matches = re.findall(r'\[(.*?)\]', text)
        
        for content in matches:
            try:
                # Try to convert the content to a number
                int(content)
                # If successful, return the original string content
                return content
            except ValueError:
                # If the conversion fails, continue to the next match
                continue
        
        return None
    item_number = extract_content(action)
    # print('extract_content', item_number)
    if (not item_number) or (f"[{item_number}]" in observation):
        return True
    return False
    

In [ ]:
file_path = 'test.jsonl'
grouped_data = group_data(file_path)
grouped_data2 = de_noise(grouped_data)

In [ ]:
with open('test.json', 'w') as f:
    json.dump(grouped_data2, f)

In [ ]:
list(grouped_data2.items())[0]